In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Task A
data = load_diabetes(as_frame=True)

X = data.data
y = data.target

display(X.head())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
X_train.shape, X_test.shape

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


((353, 10), (89, 10))

In [3]:
# Task B
rf_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=100,
                                    max_depth=5,
                                    min_samples_leaf=2,
                                    random_state=42)),
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds) ** 0.5
rf_r2 = r2_score(y_test, rf_preds)

print(f"RF MAE : {rf_mae:.4f}")
print(f"RF RMSE: {rf_rmse:.4f}")
print(f"RF R2  : {rf_r2:.4f}")

RF MAE : 43.8554
RF RMSE: 53.7739
RF R2  : 0.4542


In [4]:
# Task C
Path("../models").mkdir(parents=True, exist_ok=True)

model_path = "../models/random_forest_diabetes_pipeline.joblib"

joblib.dump(rf_pipeline, model_path)

loaded_model = joblib.load(model_path)
loaded_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

In [5]:
# Task D
preds_before = rf_pipeline.predict(X_test)
preds_after = loaded_model.predict(X_test)

np.allclose(preds_before, preds_after)

True

In [6]:
# Task E
sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)

actual = y_test.iloc[0]

print(f"Prediction: {prediction[0]:.2f}")
print(f"Actual    : {actual:.2f}")
print(f"Error     : {abs(prediction[0] - actual):.2f}")

Prediction: 156.72
Actual    : 219.00
Error     : 62.28


In [7]:
# Task F
def predict_one(model, row: pd.DataFrame) -> float:
    prediction = model.predict(row)

    return float(prediction[0])

test_pred = predict_one(rf_pipeline, sample)

test_pred

156.71823509175954

In [8]:
# Task G
manual_sample = pd.DataFrame([X_train.mean()])
display(manual_sample)

prediction = predict_one(loaded_model, manual_sample)
prediction

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.001442,0.000184,0.001736,0.001179,-0.000556,-0.000806,-0.000989,0.000377,0.001216,0.001891


131.02140672124332

## Выводы

- Модель после обучения можно сохранить в файл и использовать позже без повторного обучения.
- Важно сохранять весь pipeline, потому что он содержит preprocessing и саму модель.
- Inference — это использование обученной модели для предсказаний на новых данных.
- После загрузки модели нужно проверить, что предсказания совпадают с исходной моделью.
- Одна строка для sklearn должна быть передана как DataFrame, а не как Series.
- Такой подход является основой будущего FastAPI endpoint для ML-модели.